##SQL in R Analysis

In [ ]:
install.packages("sqldf")
install.packages("dplyr")

library(sqldf)
library(dplyr)

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

also installing the dependencies ‘gsubfn’, ‘proto’, ‘RSQLite’, ‘chron’


Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

Loading required package: gsubfn

Loading required package: proto

Warning message:
“no DISPLAY variable so Tk is not available”
Loading required package: RSQLite


Attaching package: ‘dplyr’


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union




In [ ]:
orders <- read.csv("orders_clean.csv")
customers <- read.csv("customers_clean.csv")
deliveries <- read.csv("deliveries_clean.csv")
drivers <- read.csv("drivers_clean.csv")
vehicles <- read.csv("vehicles_clean.csv")
hubs <- read.csv("hubs_clean.csv")
complaints <- read.csv("complaints_clean.csv")
incidents <- read.csv("incidents_clean.csv")
app_events <- read.csv("app_events_clean.csv")

In [ ]:
dim(orders)
dim(customers)
dim(deliveries)
dim(complaints)

[1] 1250   14

[1] 650  10

[1] 950  24

[1] 320  11

In [ ]:
complaints_by_service <- sqldf("
    SELECT
        o.service_type,
        COUNT(c.complaint_id) AS total_complaints,
        ROUND(AVG(c.compensation_amount), 2) AS avg_compensation,
        ROUND(AVG(c.resolution_days), 2) AS avg_resolution_days
    FROM orders o
    JOIN complaints c
        ON o.order_id = c.order_id
    GROUP BY o.service_type
    ORDER BY total_complaints DESC
")

complaints_by_service

service_type,total_complaints,avg_compensation,avg_resolution_days
<chr>,<int>,<dbl>,<dbl>
Passenger,84,19.73,7.74
Retail,83,17.91,7.89
Parcel,77,19.01,8.34
Business,39,20.79,7.95
Medical,37,20.00,7.57


In [ ]:
orders_by_customer_type <- sqldf("
    SELECT
        c.customer_type,
        COUNT(o.order_id) AS total_orders,
        ROUND(AVG(o.order_value), 2) AS avg_order_value,
        ROUND(SUM(o.order_value), 2) AS total_order_value
    FROM customers c
    JOIN orders o
        ON c.customer_id = o.customer_id
    GROUP BY c.customer_type
    ORDER BY total_orders DESC
")

orders_by_customer_type

customer_type,total_orders,avg_order_value,total_order_value
<chr>,<int>,<dbl>,<dbl>
Consumer,918,89.95,82578.55
SME,214,91.02,19477.22
Enterprise,118,99.64,11757.38


In [ ]:
vehicle_delivery_performance <- sqldf("
    SELECT
        v.maintenance_status,
        COUNT(d.delivery_id) AS total_deliveries,
        SUM(d.failed_delivery_flag) AS failed_deliveries,
        ROUND(AVG(d.delivery_duration_hours), 2) AS avg_delivery_duration,
        ROUND(AVG(d.customer_rating_post_delivery), 2) AS avg_customer_rating
    FROM vehicles v
    JOIN deliveries d
        ON v.vehicle_id = d.vehicle_id
    GROUP BY v.maintenance_status
    ORDER BY failed_deliveries DESC
")

vehicle_delivery_performance

maintenance_status,total_deliveries,failed_deliveries,avg_delivery_duration,avg_customer_rating
<chr>,<int>,<int>,<dbl>,<dbl>
Inrepair,254,77,11.90,3.63
Active,542,45,9.62,3.95
Scheduled,154,10,10.09,3.93
